In [1]:
from langchain.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.document import Document
from langchain.vectorstores.chroma import Chroma
from embedding import get_embedding_function
from langchain.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama

In [2]:
def load_documents():
    document_loader = TextLoader('text.txt')
    return document_loader.load()

In [3]:
def split_documents(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)

In [4]:
data = load_documents()
print("Documents Loaded")
chunks = split_documents(data)
print("Chuncks Created")

Documents Loaded
Chuncks Created


In [7]:
def calculate_chunk_ids(chunks):

    # This will create IDs like "data/monopoly.pdf:6:2"
    # Page Source : Page Number : Chunk Index

    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"

        # If the page ID is the same as the last one, increment the index.
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        # Calculate the chunk ID.
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id

        # Add it to the page meta-data.
        chunk.metadata["id"] = chunk_id

    return chunks

In [8]:
def add_to_chroma(chunks: list[Document]):
    # Load the existing database.
    db = Chroma(
        persist_directory="Database", embedding_function=get_embedding_function()
    )

    # Calculate Page IDs.
    chunks_with_ids = calculate_chunk_ids(chunks)

    # Add or Update the documents.
    existing_items = db.get(include=[])  # IDs are always included by default
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # Only add documents that don't exist in the DB.
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    if len(new_chunks):
        print(f"👉 Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)
        db.persist()
    else:
        print("✅ No new documents to add")

In [ ]:
from langchain.vectorstores import Milvus
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

def calculate_chunk_ids(chunks):
    return [
        Document(
            page_content=doc.page_content,
            metadata={**doc.metadata, "id": f"{doc.metadata['source']}-{i}"}
        ) for i, doc in enumerate(chunks)
    ]

def add_to_milvus(chunks: list[Document]):
    embedding_function = get_embedding_function()
    
    # Connect to Milvus
    connection_args = {
        "host": "localhost",
        "port": "19530",
        "user": "",  # Optional
        "password": "",  # Optional
        "secure": False
    }

    collection_name = "my_documents"
    
    # Create or connect to the vector store
    vectorstore = Milvus.from_documents(
        documents=[],
        embedding=embedding_function,
        collection_name=collection_name,
        connection_args=connection_args
    )

    chunks_with_ids = calculate_chunk_ids(chunks)
    new_chunk_ids = [chunk.metadata["id"] for chunk in chunks_with_ids]

    print(f"👉 Adding {len(chunks_with_ids)} documents to Milvus")
    vectorstore.add_documents(chunks_with_ids, ids=new_chunk_ids)


In [9]:
add_to_chroma(chunks)

C:\Users\manoj\AppData\Local\Temp\ipykernel_17268\2164321968.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


Number of existing documents in DB: 0
👉 Adding new documents: 421


C:\Users\manoj\AppData\Local\Temp\ipykernel_17268\2164321968.py:25: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


---

In [52]:
PROMPT_TEMPLATE = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the above context only. 
You should not share any information out of the knowledge of the context.
You can use your knownledge only if the question is related to the context
If the question is unrelated to the context politely decline: {question}
"""

In [53]:
def query_rag(query_text: str):
    # Prepare the DB.
    embedding_function = get_embedding_function()
    db = Chroma(persist_directory="Database", embedding_function=embedding_function)

    # Search the DB.
    print("Searching...")
    results = db.similarity_search_with_score(query_text, k=5)
    print("Search Finished")

    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context=context_text, question=query_text)
    # print(prompt)

    print("Model Generation\n")
    model = Ollama(model="stablelm-zephyr:3b")
    response_text = model.invoke(prompt)

    sources = [doc.metadata.get("id", None) for doc, _score in results]
    formatted_response = f"Response: {response_text}\nSources: {sources}"
    print(formatted_response)
    return response_text

In [54]:
question = '''A square coil of side 30 cm with 500 
turns is kept in a uniform magnetic 
field of 0.4 T. The plane of the coil is 
inclined at an angle of 30o to the field. 
Calculate the magnetic flux through 
the coil. '''
output = query_rag(question)
#print("Output: ",output)

Searching...
Search Finished
Model Generation

Response: Based only on the given context, here's how I would answer each question:

1. For question (i) - The magnetic moment of the magnet:
Magnetic moment (μ) is defined as the product of an object's magnetization and its mass. In this case, since we know the strength of the magnetic field (0.8 T), we can calculate the magnetic moment of the bar magnet by multiplying it with the mass of the magnet. The mass isn't given in the context, so I'll assume it to be a unit mass (1 kg).

μ = 1 * 0.8  ≈ 0.8 A m^2

1. For question (ii) - Calculating work done by applied force and magnetic field:
Since we know that the magnet experiences a torque due to the interaction between its magnetic moment and the magnetic field, we can use this principle to calculate both the work done by the applied force and the work done by the magnetic field in moving it from one configuration to another. The equation for work (W) is given as W = ΔU/Δt, where ΔU is the 